# 06 — Hasil akhir dan evaluasi keputusan

Saya menyimpan **submission 010 sebagai hasil utama** dan **012 sebagai eksperimen lanjutan**. Notebook ini membaca bukti tersimpan dan mengevaluasi alasan pemilihannya; notebook ini tidak melatih ulang model, tidak menghitung F1 Kaggle dari hidden test, dan tidak menyatakan bahwa audit lokal setara dengan private score.

Public Macro F1 yang dilaporkan: **010 = 0,60627** dan **012 = 0,60449** (13–14 Oktober 2026). Private leaderboard belum tersedia.

In [ ]:
from pathlib import Path
from IPython.display import display
import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "reports").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "reports").is_dir():
    raise FileNotFoundError("Jalankan dari root repository atau folder notebooks.")

history = json.loads((ROOT / "reports" / "submission10_followup_results.json").read_text(encoding="utf-8"))
metrics12 = json.loads((ROOT / "reports" / "fixed_run" / "metrics.json").read_text(encoding="utf-8"))
geo = json.loads((ROOT / "reports" / "fixed_run" / "geographic_stress.json").read_text(encoding="utf-8"))

paths = {
    "010": ROOT / "submissions" / history["submission10"]["filename"],
    "012": ROOT / "submissions" / "010_Constantine(12).csv",
}
hashes = {
    "010": history["submission10"]["sha256"],
    "012": metrics12["submission_sha256"],
}
for version, path in paths.items():
    assert hashlib.sha256(path.read_bytes()).hexdigest() == hashes[version], f"File {version} berubah."

score10 = float(history["submission10"]["public_macro_f1"])
score12 = float(metrics12["public_kaggle_score"])
assert score10 == 0.60627 and score12 == 0.60449
assert history["submission10"]["private_macro_f1"] is None
print("Checksum submission 010 dan 012 sesuai catatan; skor private belum diketahui.")

## Public leaderboard dibanding validasi lokal

Skor public merupakan angka yang dilaporkan setelah unggahan, sedangkan angka CAL/SEL/AUDIT dihitung dari sebagian data train yang memiliki label. Karena skema dan sampelnya berbeda, skor validasi historis 010 tidak dibandingkan langsung dengan AUDIT 012 sebagai uji berpasangan.

In [ ]:
results = pd.DataFrame([
    {"Submission": "010", "Posisi": "Utama", "Public Macro F1 (laporan)": score10,
     "Local Macro F1 yang tersedia": history["submission10"]["confirmation_macro_f1"],
     "Jenis evaluasi lokal": "Confirmation historis adaptif"},
    {"Submission": "012", "Posisi": "Eksperimen lanjutan", "Public Macro F1 (laporan)": score12,
     "Local Macro F1 yang tersedia": metrics12["audit_f1"],
     "Jenis evaluasi lokal": "AUDIT baru, setelah seleksi"},
])
display(results)
print(f"Selisih public (012 - 010) = {score12 - score10:+.5f}.")
print(f"Selisih audit 012 vs baseline run yang sama = "
      f"{metrics12['audit_f1'] - metrics12['audit_raw_lgb47_f1']:+.6f}.")
print("Perubahan prediksi 010 -> 012:", format(metrics12["changed_vs_submission10"], ","))
print("Skor publik 011 yang tercatat:", history["candidate11"]["public_macro_f1"],
      "(keterkaitan ke checksum CSV 011 belum terverifikasi secara independen).")

## Kesalahan klasifikasi pada AUDIT 012

Angka per kelas di bawah dihitung pada 32.009 baris AUDIT yang memiliki label aktual, **bukan pada test Kaggle**. Kelas 2 dan 3 paling sulit dibedakan. Pengujian ini mendukung analisis perilaku model 012, tetapi bukan bukti bahwa 012 akan unggul di private leaderboard.

In [ ]:
per_class = pd.DataFrame([
    {
        "Grade": int(grade),
        "Precision": metrics12["audit_classification"][grade]["precision"],
        "Recall": metrics12["audit_classification"][grade]["recall"],
        "F1": metrics12["audit_classification"][grade]["f1-score"],
        "Support": int(metrics12["audit_classification"][grade]["support"]),
    }
    for grade in ["1", "2", "3", "4", "5"]
])
assert per_class["Support"].sum() == 32009
display(per_class.set_index("Grade").round(4))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(per_class["Grade"].astype(str), per_class["F1"])
ax.set(xlabel="Damage grade", ylabel="F1 pada AUDIT 012",
       title="F1 per kelas, 32.009 baris AUDIT (bukan hidden test)", ylim=(0, 1))
for x, val in enumerate(per_class["F1"]):
    ax.text(x, val + 0.015, f"{val:.3f}", ha="center", fontsize=9)
fig.tight_layout()
plt.show()

## Catatan overfitting dan risiko geografi

Eksperimen 012 memakai FIT/CAL/SEL/AUDIT terpisah dan fitur berbasis label tetangga di-cross-fit. Pada split AUDIT, Macro F1 model terpilih mencapai 0,602359 dibanding baseline LightGBM 0,590189 pada data yang sama. Hal ini mendukung eksperimen lokal 012, **bukan membuktikan** skor private nanti.

Model tetap sensitif terhadap pola wilayah. Stress test memakai sampel 60.000 baris dan menghasilkan skor lebih rendah ketika prefix enam digit tidak bertumpang tindih. Distribusi label pada kedua holdout berbeda sehingga gap ini bukan perkiraan penurunan Kaggle secara langsung.

Untuk 010, confirmation historis dipakai selama proses seleksi adaptif dan checkpoint ensemble akhir belum lengkap. Dengan demikian, generalization gap model 010 yang tepat belum dapat dihitung ulang. Tidak ada bukti memadai untuk menyebutnya overfit berat maupun bebas overfitting.

In [ ]:
random_f1 = float(geo["splits"]["random"]["macro_f1"])
group_f1 = float(geo["splits"]["group_prefix6"]["macro_f1"])
display(pd.DataFrame([
    {"Uji geografis (sampel 60.000)": "Random split", "Macro F1": random_f1},
    {"Uji geografis (sampel 60.000)": "Group split prefix6", "Macro F1": group_f1},
]))
print(f"Selisih random - group = {random_f1 - group_f1:.6f}.")
print("Kedua holdout berbeda distribusi dan bukan estimasi langsung private score.")

decision = {
    "best_public_submission": "010_Constantine(10).csv",
    "alternative_submission": "010_Constantine(12).csv",
    "reported_public_macro_f1": {"010": score10, "012": score12},
    "public_score_source": "Laporan hasil unggahan Kaggle 13–14 Oktober 2026; tidak diambil melalui Kaggle API",
    "private_macro_f1": None,
    "local_audit_012_macro_f1": float(metrics12["audit_f1"]),
    "local_audit_012_baseline_macro_f1": float(metrics12["audit_raw_lgb47_f1"]),
    "historical_010_confirmation_macro_f1": float(history["submission10"]["confirmation_macro_f1"]),
    "historical_confirmation_adaptive": True,
    "full_training_of_010_exactly_reproduced": False,
    "differences_in_test_predictions": int(metrics12["changed_vs_submission10"]),
    "recommendation": "Gunakan 010 sebagai hasil utama; simpan 012 sebagai eksperimen lanjutan atau opsi kedua jika diizinkan.",
}
out = ROOT / "artifacts" / "submission_review"
out.mkdir(parents=True, exist_ok=True)
(out / "final_decision.json").write_text(json.dumps(decision, indent=2, ensure_ascii=False), encoding="utf-8")
print("Keputusan akhir disimpan:", out / "final_decision.json")

## Keputusan

**Submission 010 tetap menjadi pilihan utama** karena memiliki public Macro F1 tertinggi yang dilaporkan, yakni 0,60627. **Submission 012 dipertahankan sebagai kandidat kedua** dengan skor 0,60449 dan dokumentasi audit yang lebih rinci. Keduanya boleh dijadikan dua pilihan final hanya apabila aturan kompetisi memang mengizinkan dua submission final.

Tidak ada bukti skor private saat ini. Skor yang berdekatan tidak menjamin peringkat 010 dan 012 akan sama pada private leaderboard. File 010 tidak ditimpa dan notebook ini tidak mengklaim memproduksinya ulang dari raw data. Riwayat metode dan keterbatasan ada di reports/METODOLOGI.md, reports/PROVENANCE_010.md, dan reports/fixed_run/ANALYSIS.md.